In [1]:
from pathlib import Path
from statsmodels.tsa.seasonal import seasonal_decompose, STL
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
import sys
from catboost import CatBoostClassifier, Pool, EFstrType
from sklearn.metrics import roc_auc_score, log_loss, average_precision_score

# путь до корня проекта
PROJECT_ROOT = Path("../../").resolve()

# добавляем в PYTHONPATH
sys.path.append(str(PROJECT_ROOT))

from src.utils.preprocess import (
    preprocess_initial_dataset,
    split_dataset,
    remove_log_features,
    split_dataset_with_val,
)


pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

DATA_PATH = Path("../../data/cleaned_dataset/final_dataset_from_notebooks.csv")
RAW_DATA_PATH = Path("../../data/raw/clean_dataset.csv")

In [2]:
df = pd.read_csv(DATA_PATH)
df_raw = pd.read_csv(RAW_DATA_PATH)
df = df.merge(
    df_raw[
        ["row_id", "lead_Стоимость доставки", "lead_Масса (гр)", "lead_Высота"]
    ],
    on="row_id",
    how="left",
)
df = remove_log_features(df)
del df_raw
# Приводим всё к нужному типу
df = preprocess_initial_dataset(df)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17966 entries, 0 to 18787
Columns: 110 entries, contact_pvz_code to contact_has_pvz_code
dtypes: datetime64[ns](2), float64(18), int64(64), object(26)
memory usage: 15.2+ MB


C:\Users\Михаил\AppData\Local\Temp\ipykernel_17380\3613829215.py:2: DtypeWarning: Columns (79,80,81) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv(RAW_DATA_PATH)


In [3]:
print(*df.columns.tolist(), sep="\n")

contact_pvz_code
lead_weight_gm
lead_responsible_user_id
lead_utm_group
lead_utm_referrer
lead_tags
lead_utm_source
lead_Квалификация лида
is_responsible_same
has_weight
contact_region_pvz
lead_utm_referrer_site
lead_has_roistat
lead_utm_id_1
lead_utm_id_2
lead_utm_id_3
lead_utm_device_type
lead_utm_site
lead_utm_position
lead_utm_reatrgeting_id
lead_utm_region_name
lead_is_utm_campaign_type_1
contact_Город
contact_region
lead_manager_category
lead_rate_is_warehouse_to_warehouse
lead_formname_has_value
lead_has_creation_date
lead_creation_date_week
lead_creation_date_month
lead_creation_date_quarter
lead_creation_date_dayofweek
lead_creation_date_sin
lead_creation_date_cos
sale_date_month
sale_date_quarter
sale_date_dayofweek
sale_date_week
sale_date_sin
sale_date_cos
lead_items_count
lead_total_quantity
lead_total_cost_from_composition
lead_has_health_supplement
lead_has_pillow
lead_has_mattress
lead_has_brace
lead_has_footwear
lead_has_accessory
lead_has_health_product
lead_categorie

In [4]:
def evaluate(model, X, y):
    y_refuse = (y == 0).astype(int)
    proba_refuse = model.predict_proba(X)[:, 0]
    return {
        "ROC_AUC": roc_auc_score(y_refuse, proba_refuse),
        "PR_AUC": average_precision_score(y_refuse, proba_refuse),
        "LogLoss": log_loss(y, model.predict_proba(X)),
    }

Обучаем датасет на всех признаках, чтобы посмотреть, какие признаки имеют наибольшее влияние на модель и есть ли среди них утечки

In [5]:
target = "buyout_flag"

X = df.drop(columns=[target, "sale_ts"])
y = df[target]

cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

train_pool = Pool(X, y, cat_features=cat_features)

model = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.1,
    verbose=50,
    random_seed=42,
)

model.fit(train_pool)

0:	learn: 0.6411631	total: 224ms	remaining: 1m 6s
50:	learn: 0.4057866	total: 3.65s	remaining: 17.8s
100:	learn: 0.3931512	total: 7.12s	remaining: 14s
150:	learn: 0.3814602	total: 10.7s	remaining: 10.6s
200:	learn: 0.3703261	total: 14.3s	remaining: 7.06s
250:	learn: 0.3586711	total: 17.9s	remaining: 3.5s
299:	learn: 0.3510226	total: 21.5s	remaining: 0us


In [6]:
fi = pd.DataFrame(
    {
        "feature": X.columns,
        "importance": model.get_feature_importance(train_pool),
    },
).sort_values(by="importance", ascending=False)

fi.head(20)

,feature,importance
72,lead_payment_type,5.439240
66,lead_has_length,4.756968
60,lead_created_ts,4.094471
59,timedelta_between_sale_and_creation,3.492951
42,lead_total_cost_from_composition,2.681770
68,lead_price,2.625012
5,lead_tags,2.494132
76,lead_mass_known,2.469882
16,lead_utm_device_type,2.424856
75,lead_group_quality,2.422282


In [7]:
df = df.sort_values("sale_ts")

X_train, y_train, X_test, y_test = split_dataset(df)

cat_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

model = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.1,
    verbose=0,
    cat_features=cat_features,
    random_state=42,
    )


model.fit(X_train, y_train)

metrics_full  = evaluate(model, X_test, y_test)
print("TIME SPLIT:", metrics_full)

TIME SPLIT: {'ROC_AUC': 0.6638482777589708, 'PR_AUC': 0.3887107605370326, 'LogLoss': 0.4305280496738976}


Удаляем из модели субъективную оценку менеджеров

In [8]:
df_reduced = df.drop(columns=["lead_Квалификация лида"])

X_train, y_train, X_test, y_test = split_dataset(df_reduced)

cat_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

model = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.1,
    verbose=0,
    cat_features=cat_features,
    random_state=42,
    )


model.fit(X_train, y_train)

metrics_reduced  = evaluate(model, X_test, y_test)
print("TIME SPLIT:", metrics_reduced)

TIME SPLIT: {'ROC_AUC': 0.6595262779282329, 'PR_AUC': 0.36881750358120025, 'LogLoss': 0.4403641513824959}


In [9]:
train_pool = Pool(
    data=X_train,
    label=y_train,
    cat_features=cat_features
)

importances = model.get_feature_importance(
    data=train_pool,
    type=EFstrType.FeatureImportance
)

fi = pd.DataFrame({
    "feature": X_train.columns,
    "importance": importances
}).sort_values(by="importance", ascending=False)

fi.head(20)

,feature,importance
67,lead_price,3.486347
71,lead_payment_type,3.128050
41,lead_total_cost_from_composition,3.083818
5,lead_tags,3.033921
58,timedelta_between_sale_and_creation,2.967541
59,lead_created_ts,2.864561
15,lead_utm_device_type,2.724714
74,lead_group_quality,2.588464
12,lead_utm_id_1,2.246633
105,lead_Высота,2.045763


Удаляем lead_group_quality так как потенциально содержит данные из будущего

In [10]:
df_reduced = df_reduced.drop(columns=["lead_group_quality"])

X_train, y_train, X_val, y_val, X_test, y_test = split_dataset_with_val(df_reduced)

cat_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

model = CatBoostClassifier(
    iterations=2000,
    depth=6,
    learning_rate=0.1,
    verbose=100,
    cat_features=cat_features,
    random_state=42,
    class_weights={0: 1, 1: 5},
    eval_metric="AUC",
    early_stopping_rounds=100,
)


model.fit(X_train, y_train, eval_set=(X_val, y_val))

metrics_reduced = evaluate(model, X_test, y_test)
print("TIME SPLIT:", metrics_reduced)

0:	test: 0.4948507	best: 0.4948507 (0)	total: 58.3ms	remaining: 1m 56s
100:	test: 0.6347789	best: 0.6347789 (100)	total: 5.67s	remaining: 1m 46s
200:	test: 0.6358455	best: 0.6363337 (167)	total: 11.6s	remaining: 1m 44s
300:	test: 0.6377640	best: 0.6393592 (285)	total: 17.9s	remaining: 1m 40s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.639359198
bestIteration = 285

Shrink model to first 286 iterations.
TIME SPLIT: {'ROC_AUC': 0.678800461238998, 'PR_AUC': 0.4062804622876623, 'LogLoss': 0.46344676536329676}


In [11]:
train_pool = Pool(
    data=X_train,
    label=y_train,
    cat_features=cat_features
)

importances = model.get_feature_importance(
    data=train_pool,
    type=EFstrType.FeatureImportance
)

fi = pd.DataFrame({
    "feature": X_train.columns,
    "importance": importances
}).sort_values(by="importance", ascending=False)

fi.head(20)

,feature,importance
71,lead_payment_type,11.889413
59,lead_created_ts,4.483890
104,lead_Высота,4.109042
15,lead_utm_device_type,3.778163
99,lead_height_known,3.052347
75,contact_to_lead_hours,2.867246
3,lead_utm_group,2.770163
84,lead_utm_campaign_missing,2.553224
67,lead_price,2.546965
6,lead_utm_source,2.138551
